# Phase I · Velocity time series: prediction vs. ground truth

**Question:** *when* does the Radarize Translation model (`transnet18`, ResNet-18) predict velocity well or badly within one sequence?

One interactive view with a shared time axis:

| Row | Shows |
|---|---|
| Overview | error magnitude ‖e‖ = √(eₓ² + e_y²) over the whole sequence. **Drag to zoom** all rows; double-click to reset |
| vx | ground truth (grey) and prediction (blue), m/s |
| eₓ | signed error eₓ = predicted − true vx |
| vy | ground truth and prediction |
| e_y | signed error e_y = predicted − true vy |
| Selected frames | values of the frames you clicked |

**Hover** shows every value at the nearest frame. **Click** marks a frame (orange) in all rows and adds it to the table; click it again to unmark; **double-click** clears all.

Both error rows use the same y-scale, so eₓ and e_y are directly comparable. The two velocity rows have their own y-scales, because vy is much smaller than vx for a robot.

**Input:** `results/phase1/main_0/<sequence>_predictions.npz` with `time`, `flow_pred_xs`, `flow_pred_ys`, `flow_gt_xs`, `flow_gt_ys` (the layout written by Radarize's `tools/test_flow.py`).

**Requires:** `pip install altair` in the RadaVis environment (pure Python, ~1 MB; Altair 6.0 is the newest release for Python 3.9). The chart loads Vega from a CDN, so it needs an internet connection to display.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import altair as alt
except ImportError as e:
    raise ImportError("Altair is missing. In the RadaVis environment run:  pip install altair") from e

print("altair", alt.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

altair 6.0.0 | pandas 2.2.3 | numpy 1.26.4


## 1 · Configuration

In [2]:
SEQUENCE = "bot_beckman_5thfloor_1"
SPLIT = "main_0"          # which checkpoint produced the predictions (output_main_0/transnet18)


def find_repo_root(start: Path) -> Path:
    # Walk up from the notebook's folder to the RadaVis root (the folder holding results/ and notebooks/).
    for p in [start, *start.parents]:
        if (p / "results").is_dir() and (p / "notebooks").is_dir():
            return p
    raise FileNotFoundError("RadaVis root not found above %s" % start)


ROOT = find_repo_root(Path.cwd().resolve())
NPZ_PATH = ROOT / "results" / "phase1" / SPLIT / f"{SEQUENCE}_predictions.npz"
HTML_OUT = NPZ_PATH.with_name(f"{SEQUENCE}_timeseries.html")
SPLIT_CONFIG = ROOT / "radarize_ae" / "configs" / f"{SPLIT}.yaml"   # optional: used for the held-out check

print("RadaVis root:", ROOT)
print("Predictions :", NPZ_PATH)

RadaVis root: F:\visualization\RadaVis
Predictions : F:\visualization\RadaVis\results\phase1\main_0\bot_beckman_5thfloor_1_predictions.npz


## 2 · Load and sanity-check the predictions

These checks run before plotting, because a time-axis or length problem would make every plot misleading.

Two checks matter in particular:
- **Held-out sequence:** the sequence must be in the split's `TEST_SPLIT`. Otherwise the checkpoint was trained on it and its errors look too good.
- **Frame spacing:** irregular gaps change how the time axis should be read.

In [3]:
REQUIRED = ["time", "flow_pred_xs", "flow_pred_ys", "flow_gt_xs", "flow_gt_ys"]

with np.load(NPZ_PATH, allow_pickle=False) as npz:
    missing = [k for k in REQUIRED if k not in npz.files]
    if missing:
        raise KeyError(f"{NPZ_PATH.name} lacks {missing}; found {npz.files}")
    raw = {k: np.asarray(npz[k], dtype=np.float64).ravel() for k in REQUIRED}

n = len(raw["time"])
lengths = {k: len(v) for k, v in raw.items()}
assert len(set(lengths.values())) == 1, f"arrays differ in length: {lengths}"
for k, v in raw.items():
    assert np.isfinite(v).all(), f"{k} contains NaN/inf"

dt = np.diff(raw["time"])
assert (dt > 0).all(), "time is not strictly increasing"
median_dt = float(np.median(dt))
long_gaps = int((dt > 1.5 * median_dt).sum())

print(f"frames          : {n}")
print(f"duration        : {raw['time'][-1] - raw['time'][0]:.2f} s")
print(f"frame spacing   : median {median_dt * 1000:.1f} ms ({1 / median_dt:.1f} Hz), "
      f"min {dt.min() * 1000:.1f} ms, max {dt.max() * 1000:.1f} ms")
print(f"gaps > 1.5×median: {long_gaps} (time axis is real time, so these show as slightly wider steps)")


def held_out_sequences(cfg_path: Path):
    # Read TEST_SPLIT from a Radarize split config without needing PyYAML.
    seqs, in_test = [], False
    for line in cfg_path.read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith("TEST_SPLIT:"):
            in_test = True
            continue
        if in_test:
            if stripped.startswith("- "):
                seqs.append(stripped[2:].strip())
            elif stripped:
                break
    return seqs


if SPLIT_CONFIG.exists():
    test_split = held_out_sequences(SPLIT_CONFIG)
    if SEQUENCE in test_split:
        print(f"held-out check  : OK, {SEQUENCE} is in {SPLIT}'s TEST_SPLIT ({len(test_split)} test sequences)")
    else:
        print(f"held-out check  : WARNING, {SEQUENCE} is NOT in {SPLIT}'s TEST_SPLIT. "
              "This checkpoint saw it during training/validation.")
else:
    print(f"held-out check  : skipped ({SPLIT_CONFIG} not found)")

frames          : 1798
duration        : 59.90 s
frame spacing   : median 33.3 ms (30.0 Hz), min 15.8 ms, max 50.6 ms
gaps > 1.5×median: 20 (time axis is real time, so these show as slightly wider steps)
held-out check  : OK, bot_beckman_5thfloor_1 is in main_0's TEST_SPLIT (29 test sequences)


## 3 · One row per frame

`t` is seconds since the first frame. Errors are **predicted − true**, so a positive eₓ means the model over-estimates forward velocity.

In [4]:
df = pd.DataFrame({
    "frame": np.arange(n),
    "t": raw["time"] - raw["time"][0],
    "gt_vx": raw["flow_gt_xs"],
    "pred_vx": raw["flow_pred_xs"],
    "gt_vy": raw["flow_gt_ys"],
    "pred_vy": raw["flow_pred_ys"],
})
df["err_x"] = df["pred_vx"] - df["gt_vx"]
df["err_y"] = df["pred_vy"] - df["gt_vy"]
df["err_norm"] = np.hypot(df["err_x"], df["err_y"])
df["gt_speed"] = np.hypot(df["gt_vx"], df["gt_vy"])

summary = pd.DataFrame({
    "MAE": [df.err_x.abs().mean(), df.err_y.abs().mean()],
    "RMSE": [np.sqrt((df.err_x ** 2).mean()), np.sqrt((df.err_y ** 2).mean())],
    "mean error (bias)": [df.err_x.mean(), df.err_y.mean()],
    "true range": [f"{df.gt_vx.min():.3f} … {df.gt_vx.max():.3f}", f"{df.gt_vy.min():.3f} … {df.gt_vy.max():.3f}"],
    "predicted range": [f"{df.pred_vx.min():.3f} … {df.pred_vx.max():.3f}", f"{df.pred_vy.min():.3f} … {df.pred_vy.max():.3f}"],
}, index=["vx (m/s)", "vy (m/s)"])
display(summary.round(4))
df.head()

,MAE,RMSE,mean error (bias),true range,predicted range
vx (m/s),0.0484,0.0667,-0.0022,-0.118 … 0.425,-0.031 … 0.314
vy (m/s),0.0250,0.0346,0.0059,-0.145 … 0.143,-0.045 … 0.059


,frame,t,gt_vx,pred_vx,gt_vy,pred_vy,err_x,err_y,err_norm,gt_speed
0,0,0.000000,0.236520,0.245407,0.001114,0.021701,0.008888,0.020587,0.022423,0.236522
1,1,0.033394,0.236520,0.238459,0.001114,0.017843,0.001939,0.016729,0.016841,0.236522
2,2,0.066673,0.301143,0.238281,0.011465,0.018246,-0.062862,0.006781,0.063227,0.301361
3,3,0.100035,0.227810,0.252268,-0.000637,0.027808,0.024458,0.028445,0.037515,0.227811
4,4,0.133355,0.227810,0.244235,-0.000637,0.019147,0.016425,0.019785,0.025714,0.227811


## 4 · Interactive time series

In [5]:
# Colours: ground truth is a neutral reference line; prediction and the selection highlight are
# categorical slots 1 and 2 of the RadaVis palette (checked for colour-vision deficiency).
INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
GT_COLOR, PRED_COLOR, SELECT_COLOR = INK_2, "#2a78d6", "#eb6834"

WIDTH = 920
F3 = ".3f"
TOOLTIP = [
    alt.Tooltip("frame:Q", title="frame"),
    alt.Tooltip("t:Q", title="t (s)", format=".2f"),
    alt.Tooltip("gt_vx:Q", title="true vx", format=F3),
    alt.Tooltip("pred_vx:Q", title="pred vx", format=F3),
    alt.Tooltip("err_x:Q", title="error x", format="+.3f"),
    alt.Tooltip("gt_vy:Q", title="true vy", format=F3),
    alt.Tooltip("pred_vy:Q", title="pred vy", format=F3),
    alt.Tooltip("err_y:Q", title="error y", format="+.3f"),
    alt.Tooltip("err_norm:Q", title="|error|", format=F3),
]


def symmetric_domain(values, pad=1.05):
    m = float(np.max(np.abs(values))) * pad
    return [-m, m]


def make_timeseries_chart(df: pd.DataFrame, title: str) -> alt.VConcatChart:
    base = alt.Chart(df)

    # Interactions --------------------------------------------------------------
    brush = alt.selection_interval(name="zoom", encodings=["x"])
    hover = alt.selection_point(name="hover", fields=["frame"], nearest=True,
                                on="mouseover", clear="mouseout", empty=False)
    picked = alt.selection_point(name="picked", fields=["frame"], nearest=True,
                                 on="click", toggle="true", clear="dblclick", empty=False)

    x_detail = alt.X("t:Q", title=None, scale=alt.Scale(domain=brush, nice=False),
                     axis=alt.Axis(labels=False, ticks=False, domain=False))
    x_bottom = alt.X("t:Q", title="time since first frame (s)", scale=alt.Scale(domain=brush, nice=False))

    def frame_markers(x_enc, show_hover=True):
        # Vertical rules for selected frames (+ hover crosshair) and an invisible nearest-frame hit layer.
        layers = [
            base.mark_rule(color=SELECT_COLOR, strokeWidth=1.5, clip=True).encode(x=x_enc).transform_filter(picked),
        ]
        if show_hover:
            layers.append(base.mark_rule(color=MUTED, strokeWidth=1, clip=True).encode(x=x_enc).transform_filter(hover))
        layers.append(
            base.mark_point(opacity=0, size=60, clip=True).encode(x=x_enc, tooltip=TOOLTIP).add_params(hover, picked)
        )
        return layers

    def velocity_panel(gt_col, pred_col, label, x_enc):
        lines = (
            base.transform_fold([gt_col, pred_col], as_=["series_key", "value"])
            .transform_calculate(series=f"datum.series_key === '{gt_col}' ? 'ground truth' : 'prediction'")
            .mark_line(strokeWidth=1.5, interpolate="linear", clip=True)
            .encode(
                x=x_enc,
                y=alt.Y("value:Q", title=f"{label} (m/s)"),
                color=alt.Color("series:N", title=None,
                                scale=alt.Scale(domain=["ground truth", "prediction"], range=[GT_COLOR, PRED_COLOR]),
                                legend=alt.Legend(orient="top", direction="horizontal", symbolType="stroke",
                                                  symbolStrokeWidth=2)),
            )
        )
        zero = base.mark_rule(color=AXIS, strokeWidth=1).encode(y=alt.datum(0))
        return alt.layer(zero, lines, *frame_markers(x_enc)).properties(width=WIDTH, height=150)

    err_domain = symmetric_domain(np.concatenate([df["err_x"], df["err_y"]]))

    def error_panel(col, label, x_enc):
        area = base.mark_area(color=INK_2, opacity=0.18, clip=True).encode(
            x=x_enc, y=alt.Y(f"{col}:Q", title=f"{label} (m/s)", scale=alt.Scale(domain=err_domain)), y2=alt.datum(0))
        line = base.mark_line(color=INK_2, strokeWidth=1, clip=True).encode(x=x_enc, y=f"{col}:Q")
        zero = base.mark_rule(color=AXIS, strokeWidth=1).encode(y=alt.datum(0))
        return alt.layer(zero, area, line, *frame_markers(x_enc)).properties(width=WIDTH, height=90)

    overview = alt.layer(
        base.mark_area(color=INK_2, opacity=0.35).encode(
            x=alt.X("t:Q", title=None, scale=alt.Scale(nice=False), axis=alt.Axis(labels=True, tickCount=12)),
            y=alt.Y("err_norm:Q", title="|error|", axis=alt.Axis(tickCount=2))),
        base.mark_rule(color=SELECT_COLOR, strokeWidth=1).encode(x="t:Q").transform_filter(picked),
    ).add_params(brush).properties(
        width=WIDTH, height=55,
        title=alt.TitleParams("Overview: drag to zoom all rows, double-click to reset", fontSize=11,
                              fontWeight="normal", color=INK_2, anchor="start"))

    table = (
        base.transform_filter(picked)
        .transform_calculate(
            frame_s="format(datum.frame, 'd')", t_s="format(datum.t, '.2f')",
            true_vx="format(datum.gt_vx, '.3f')", pred_vx_s="format(datum.pred_vx, '.3f')",
            error_x="format(datum.err_x, '+.3f')",
            true_vy="format(datum.gt_vy, '.3f')", pred_vy_s="format(datum.pred_vy, '.3f')",
            error_y="format(datum.err_y, '+.3f')", abs_error="format(datum.err_norm, '.3f')")
        .transform_window(rank="row_number()", sort=[alt.SortField("frame")])
        .transform_filter("datum.rank <= 25")
        .transform_fold(["frame_s", "t_s", "true_vx", "pred_vx_s", "error_x", "true_vy", "pred_vy_s", "error_y", "abs_error"],
                        as_=["column", "text"])
        .transform_calculate(column_label=(
            "{frame_s: 'frame', t_s: 't (s)', true_vx: 'true vx', pred_vx_s: 'pred vx', error_x: 'error x', "
            "true_vy: 'true vy', pred_vy_s: 'pred vy', error_y: 'error y', abs_error: '|error|'}[datum.column]"))
        .mark_text(align="right", baseline="middle", dx=30, fontSize=11, color=INK)
        .encode(
            x=alt.X("column_label:N", title=None,
                    sort=["frame", "t (s)", "true vx", "pred vx", "error x", "true vy", "pred vy", "error y", "|error|"],
                    axis=alt.Axis(orient="top", labelAngle=0, labelColor=INK_2, domain=False, ticks=False)),
            y=alt.Y("frame:O", title=None, axis=None),
            text="text:N",
        )
        .properties(width=WIDTH * 0.75, title=alt.TitleParams(
            "Selected frames (first 25, by frame): click frames above to add or remove", fontSize=11,
            fontWeight="normal", color=INK_2, anchor="start"))
    )

    chart = alt.vconcat(
        overview,
        velocity_panel("gt_vx", "pred_vx", "vx", x_detail),
        error_panel("err_x", "error x", x_detail),
        velocity_panel("gt_vy", "pred_vy", "vy", x_detail),
        error_panel("err_y", "error y", x_bottom),
        table,
        spacing=8,
    ).properties(
        # resize=True: re-measure the layout when the selected-frames table grows or shrinks
        autosize=alt.AutoSizeParams(type="pad", resize=True),
        title=alt.TitleParams(title, anchor="start", fontSize=14, color=INK,
                              subtitle="hover for values · click to mark frames · error = predicted − true",
                              subtitleColor=INK_2)
    ).configure(
        background=SURFACE, font="system-ui, -apple-system, Segoe UI, sans-serif",
    ).configure_view(stroke=None).configure_axis(
        gridColor=GRID, domainColor=AXIS, tickColor=AXIS, labelColor=INK_2, titleColor=INK_2,
        labelFontSize=10, titleFontSize=11, titleFontWeight="normal",
    ).configure_legend(labelColor=INK_2, labelFontSize=11)
    return chart


import warnings
with warnings.catch_warnings():
    # The hover/click selections are deliberately shared by all rows; Altair 6 notes that it merged them.
    warnings.filterwarnings("ignore", message="Automatically deduplicated selection parameter")
    chart = make_timeseries_chart(df, f"{SEQUENCE}: Translation model (transnet18, {SPLIT}), {len(df)} frames")
chart

alt.VConcatChart(...)

## 5 · Save a standalone HTML copy

Saves an HTML file next to the predictions, so the chart opens in any browser (needs internet for the Vega scripts) and can be shared without running the notebook.

In [6]:
chart.save(str(HTML_OUT))
print("saved", HTML_OUT)

saved F:\visualization\RadaVis\results\phase1\main_0\bot_beckman_5thfloor_1_timeseries.html
